# IBM watsonx.ai Runtime - Decision Optimization API を使用したLP問題の解決

## 概要

このNotebookは、IBM watsonx.ai RuntimeのDecision Optimization APIを使用して、線形計画問題（Linear Programming: LP）を解くための完全なワークフローを提供します。

## 主な機能

- **圧縮ファイル対応**: .lp.gz形式のファイルをサポート
- **CPLEXソルバー**: IBM CPLEX 22.1を使用した最適化
- **バッチ処理**: 非同期でのジョブ実行
- **結果取得**: XML形式での最適化結果の取得

## 処理フロー

```
1. 環境セットアップ
   ↓
2. モデル登録
   ↓
3. デプロイメント作成
   ↓
4. LPファイルアップロード
   ↓
5. 最適化ジョブ実行
   ↓
6. ジョブ監視
   ↓
7. 結果取得
   ↓
8. リソースクリーンアップ
```

## 前提条件

- IBM Cloud アカウント
- watsonx.ai Runtime サービスのインスタンス
- APIキーとSpace IDの取得
- 解きたいLP問題のファイル（.lp または .lp.gz形式）
- model_dummy.zip（デプロイメント用のダミーファイル）

---

## [1] 環境セットアップ

### 1.1 ライブラリのインストール

IBM watsonx.ai Runtime Python SDKをインストールします。

**注意**: 初回実行時のみ必要です。既にインストール済みの場合はスキップできます。

In [1]:
# IBM watsonx.ai Runtime Python SDKのインストール（初回のみ）
# 実行する場合は、先頭の#を削除してください
#!pip install ibm-watsonx-ai 

### 1.2 必要なライブラリのインポート

プログラムで使用する各ライブラリをインポートします。

In [2]:
# 必要なモジュールのインポート
from ibm_watsonx_ai import APIClient, Credentials  # Watson ML APIクライアント
import base64  # Base64エンコード/デコード用（結果ファイルの取得に使用）
import time    # ジョブ監視のための待機処理用
import json    # JSON形式のデータ処理用

print("[OK] ライブラリのインポートが完了しました")

[OK] ライブラリのインポートが完了しました


---

## [2] IBM Cloud認証情報の設定

### セキュリティ警告

**重要**: 本番環境では、以下のセキュリティベストプラクティスに従ってください：

1. **APIキーをコードに直接記述しない**
   - 環境変数を使用: `os.environ.get('IBM_API_KEY')`
   - 設定ファイルを使用: `.env`ファイル + `python-dotenv`
   - シークレット管理サービスを使用: IBM Cloud Secrets Manager等

2. **バージョン管理から除外**
   - `.gitignore`にAPIキーを含むファイルを追加
   - Notebookを共有する前に必ずAPIキーを削除

3. **APIキーの定期的なローテーション**
   - 定期的にAPIキーを再生成
   - 不要になったAPIキーは即座に削除

### 必要な情報

| 項目 | 説明 | 取得方法 |
|------|------|----------|
| **api_key** | IBM Cloud APIキー（IAM APIキー） | IBM Cloud コンソール > 管理 > アクセス(IAM) > APIキー |
| **instance_url** | Watson MLサービスのエンドポイントURL | リージョンに応じて選択（下記参照） |
| **space_id** | デプロイメントスペースID | Watson Studio > デプロイメントスペース > 設定 |

### リージョン別エンドポイントURL

- **米国南部（ダラス）**: `https://us-south.ml.cloud.ibm.com`
- **東京**: `https://jp-tok.ml.cloud.ibm.com`
- **ロンドン**: `https://eu-gb.ml.cloud.ibm.com`
- **フランクフルト**: `https://eu-de.ml.cloud.ibm.com`

In [ ]:
# IBM Cloud認証情報の設定
# [警告] 以下の値を実際の認証情報に置き換えてください

api_key = "YOUR_API_KEY"  # IBM Cloud APIキー
instance_url = "https://jp-tok.ml.cloud.ibm.com"  # Watson MLインスタンスのURL
space_id = "YOUR_SPACE_ID"  # デプロイメントスペースID

# 認証情報の検証
if api_key == "YOUR_API_KEY" or space_id == "YOUR_SPACE_ID":
    print("[警告] APIキーとSpace IDを実際の値に置き換えてください")
else:
    print("[OK] 認証情報が設定されました")

[OK] 認証情報が設定されました


### 2.1 APIクライアントの初期化

設定した認証情報を使用して、Watson ML APIクライアントを初期化します。

In [4]:
# 認証情報オブジェクトの作成
credentials = Credentials(api_key=api_key, url=instance_url)

# APIクライアントの初期化（指定されたスペースIDに接続）
client = APIClient(credentials, space_id=space_id)

print("[OK] Watson ML APIクライアントの初期化が完了しました")
print(f"  接続先: {instance_url}")
print(f"  スペースID: {space_id}")

[OK] Watson ML APIクライアントの初期化が完了しました
  接続先: https://jp-tok.ml.cloud.ibm.com
  スペースID: 53e7afa7-f480-4932-b8e1-62fa18a4a002


---

## [3] ダミーモデルファイルの登録

### モデル登録の目的

watsonx.ai Runtimeでは、最適化問題を実行するために、まずモデルをデプロイメントスペースに登録する必要があります。

### モデルメタデータの説明

| メタデータ | 値 | 説明 |
|-----------|-----|------|
| **NAME** | "model_dummy.zip" | ダミーファイルの識別名 |
| **TYPE** | "do-cplex_22.1" | CPLEX 22.1を使用するDecision Optimizationモデル |
| **SOFTWARE_SPEC_ID** | "do_22.1" | Decision Optimization 22.1のランタイム環境 |

### ダミーファイルについて

実際のLPファイルは後でデータアセットとしてアップロードするため、ここではダミーのzipファイル（`model_dummy.zip`）を登録します。これはデプロイメントを作成するためのプレースホルダーとして機能します。

In [5]:
# モデルメタデータの定義
metadata = {
    # モデル名の設定
    client.repository.ModelMetaNames.NAME: "model_dummy.zip",
    
    # モデルタイプの指定: CPLEX 22.1を使用するDecision Optimizationモデル
    client.repository.ModelMetaNames.TYPE: "do-cplex_22.1",
    
    # ソフトウェア仕様IDの取得: Decision Optimization 22.1の実行環境を指定
    client.repository.ModelMetaNames.SOFTWARE_SPEC_ID: 
        client.software_specifications.get_id_by_name("do_22.1")
}

print("[OK] モデルメタデータを定義しました")

[OK] モデルメタデータを定義しました


In [6]:
# ダミーモデルファイル(model_dummy.zip)をデプロイメントスペースに保存
# 注意: 実際のLPファイルは後でデータアセットとしてアップロードされます
#       ここではデプロイメントを作成するためのプレースホルダーとして使用
from typing import Any


model_details: dict[Any, Any] = client.repository.store_model(
    model="model_dummy.zip",  # ダミーのzipファイル
    meta_props=metadata
)

# 保存されたモデルのIDを取得
model_id = client.repository.get_model_id(model_details)

print("[OK] モデルがデプロイメントスペースに登録されました")
print(f"  Model ID: {model_id}")

[OK] モデルがデプロイメントスペースに登録されました
  Model ID: 6eb66869-f165-4cb7-b40e-794472cf3b22


---

## [4] デプロイメントの作成

### デプロイメントとは

デプロイメントは、登録したモデルを実行可能な状態にするための設定です。デプロイメントを作成することで、最適化ジョブを投入できるようになります。

### デプロイメント設定の詳細

#### NAME
デプロイメントの識別名です。複数のデプロイメントを管理する際に使用します。

#### BATCH
Decision Optimizationでは、バッチデプロイメント（非同期実行）のみがサポートされています。
- **バッチ**: ジョブをキューに投入し、完了を待つ
- **注意**: リアルタイムデプロイメントは選択できません

#### HARDWARE_SPEC
使用するハードウェアリソースを指定します。

| サイズ | CPU | メモリ | 適用問題 |
|--------|-----|--------|----------|
| **S (Small)** | 2 vCPU | 8 GB | 小規模問題 |
| **M (Medium)** | 4 vCPU | 16 GB | 中規模問題 |
| **L (Large)** | 8 vCPU | 32 GB | 大規模問題 |
| **XL (Extra Large)** | 16 vCPU | 64 GB | 特大規模問題 |

**注意**: 適切なハードウェアサイズは、問題の複雑さ（変数数、制約数、整数変数の有無など）、メモリ使用量、計算時間の要件によって異なります。小さいサイズから始めて、必要に応じてスケールアップすることを推奨します。

**num_nodes**: 使用するノード数（通常は1）

In [7]:
# デプロイメントの作成
deployment_details = client.deployments.create(
    model_id,  # 先ほど登録したモデルのID
    meta_props={
        # デプロイメント名の設定
        client.deployments.ConfigurationMetaNames.NAME: "LP Deployment",
        
        # バッチデプロイメントとして設定（Decision Optimizationではバッチのみサポート）
        client.deployments.ConfigurationMetaNames.BATCH: {},
        
        # ハードウェア仕様: Sサイズ、1ノード
        # Sサイズは小規模な最適化問題に適しています
        # 問題の規模に応じて "M" や "L" に変更可能
        client.deployments.ConfigurationMetaNames.HARDWARE_SPEC: {
            "name": "S",      # ハードウェアサイズ（S/M/L/XL）
            "num_nodes": 1    # 使用するノード数
        }
    }
)

# デプロイメントIDの取得
deployment_id = client.deployments.get_id(deployment_details)

print("[OK] デプロイメントが作成されました")
print(f"  Deployment ID: {deployment_id}")
print(f"  ハードウェア: Sサイズ (2 vCPU, 8 GB)")



######################################################################################

Synchronous deployment creation for id: '6eb66869-f165-4cb7-b40e-794472cf3b22' started

######################################################################################


ready.


-----------------------------------------------------------------------------------------------
Successfully finished deployment creation, deployment_id='0711fe77-8b26-4530-a507-7829f5be6bce'
-----------------------------------------------------------------------------------------------


[OK] デプロイメントが作成されました
  Deployment ID: 0711fe77-8b26-4530-a507-7829f5be6bce
  ハードウェア: Sサイズ (2 vCPU, 8 GB)


In [8]:
import pprint
pprint.pprint(deployment_details)

{'entity': {'asset': {'id': '6eb66869-f165-4cb7-b40e-794472cf3b22'},
            'batch': {},
            'chat_enabled': False,
            'custom': {},
            'deployed_asset_type': 'do',
            'hardware_spec': {'name': 'S', 'num_nodes': 1},
            'status': {'state': 'ready'}},
 'metadata': {'created_at': '2025-12-26T08:25:59.586Z',
              'id': '0711fe77-8b26-4530-a507-7829f5be6bce',
              'modified_at': '2025-12-26T08:25:59.586Z',
              'name': 'LP Deployment',
              'owner': 'JUrymLFGan-03df8278-ddcd-4991-9f70-66ad81afaef9',
              'space_id': '53e7afa7-f480-4932-b8e1-62fa18a4a002'}}


---

## [5] LPファイルのアップロード

### データアセットとは

データアセットは、watsonx.ai Runtimeのスペース内に保存されるデータファイルです。アセットIDを使用して、ジョブ実行時に参照できます。

#### 圧縮のメリット
1. **アップロード時間の短縮**: ファイルサイズが小さくなる
2. **ストレージコストの削減**: 保存容量が減る
3. **ネットワーク帯域の節約**: 転送データ量が減る

**推奨**: 大きなファイル（> 1 MB）の場合は、必ず.gz形式で圧縮してください。

### ファイルの準備方法

```bash
# Linuxまたはmacの場合
gzip model.lp

# Windowsの場合（7-Zipを使用）
7z a -tgzip model.lp.gz model.lp
```

In [9]:
# LPファイル（gz圧縮形式）をデータアセットとしてアップロード
from typing import Any


model_details = client.data_assets.create(
    name="model.lp.gz",     # データアセット名（モデル名と区別するため"input_"を付与）
    file_path="model.lp.gz"    # ローカルファイルパス（圧縮されたLPファイル）
)

# アップロードされたデータアセットのIDを取得
# このIDは後でジョブ実行時に入力ファイルとして参照されます
model_da_id = model_details["metadata"]["asset_id"]

print("[OK] LPファイルがアップロードされました")
print(f"  Data Asset ID: {model_da_id}")
print(f"  データアセット名: model.lp.gz")
print(f"  ファイル名: model.lp.gz")

Creating data asset...
SUCCESS
[OK] LPファイルがアップロードされました
  Data Asset ID: 38eb657e-a734-4446-9d1d-911189e2d21f
  データアセット名: model.lp.gz
  ファイル名: model.lp.gz


---

## [6] 最適化ジョブの実行

### ジョブペイロードの構造

ジョブペイロードは、最適化ジョブの実行に必要なすべての設定を含むJSON形式のデータです。

### solve_parameters（解法パラメータ）

CPLEXソルバーの動作を制御するパラメータです。

#### oaas.logTailEnabled
- **値**: `"true"` または `"false"`
- **説明**: CPLEXのログ出力を有効化
- **用途**: デバッグや進捗確認に使用

#### oaas.logAttachmentName
- **値**: `"log.txt"`
- **説明**: ログファイル名

#### oaas.resultsFormat
- **値**: `"XML"`
- **説明**: 結果ファイルのフォーマット


### INPUT_DATA_REFERENCES（入力データ）

最適化問題の入力ファイルを指定します。

| フィールド | 説明 | 例 |
|-----------|------|----|
| **id** | 任意だがアップロードしたファイル名がわかりやすい | "model.lp.gz" |
| **type** | データの種類 | **"data_asset"** |
| **location.href** | データアセットのURL | "/v2/assets/{asset_id}?space_id={space_id}" |


### OUTPUT_DATA_REFERENCES（出力データ）

最適化結果の出力先を指定します。

| フィールド | 説明 | 例 |
|-----------|------|----|
| **id** | ファイルの識別子 | "solution.xml" |
| **type** | データの種類 | "data_asset" |
| **location.name** | 出力ファイル名 | "solution.xml" |

**注意**: `location.name`を指定すると、新しいデータアセットとして保存されます。

In [10]:
# ジョブペイロードの作成
solve_payload = {
    # ========================================
    # 解法パラメータの設定
    # ========================================
    "solve_parameters": {
        # CPLEXソルバーのログ出力を有効化（デバッグや進捗確認に有用）
        "oaas.logTailEnabled": "true",
        # CPLEXソルバーログ全体をジョブ出力のアセットとして保存する
        "oaas.logAttachmentName":"log.txt",
        # 結果ファイルのフォーマットをXMLに指定
        "oaas.resultsFormat": "XML"
    },

    # ========================================
    # 入力データの指定
    # ========================================
    client.deployments.DecisionOptimizationMetaNames.INPUT_DATA_REFERENCES: [
        {
            # アップロードした入力ファイル名
            "id": "model.lp.gz",
            # データアセットとして参照
            "type": "data_asset",
            # アップロードしたLPファイルのアセットIDを指定
            "location": {
                "href": f"/v2/assets/{model_da_id}?space_id={space_id}"
            }
        }
    ],

    # ========================================
    # 出力データの指定
    # ========================================
    client.deployments.DecisionOptimizationMetaNames.OUTPUT_DATA_REFERENCES: [
        {
            # 結果出力ファイルの識別子
            "id": "solution.xml",
            # データアセットとして保存
            "type": "data_asset",
            # 出力ファイル名を指定（新しいデータアセットとして作成される）
            "location": {
                "name": "solution.xml"
            }
        },
        {
            # ログ出力ファイルの識別子
            "id": "log.txt",
            # データアセットとして保存
            "type": "data_asset",
            # 出力ファイル名を指定（新しいデータアセットとして作成される）
            "location": {
                "name": "log.txt"
            }
        }    ]
}

print("[OK] ジョブペイロードを作成しました")
print("  入力: model.lp.gz (データアセット)")
print("  出力: solution.xml (XML形式)")
print("  ログ出力: 有効")

[OK] ジョブペイロードを作成しました
  入力: model.lp.gz (データアセット)
  出力: solution.xml (XML形式)
  ログ出力: 有効


In [11]:
# ジョブの投入
job_details = client.deployments.create_job(deployment_id, solve_payload)

# ジョブIDを取得
job_id = client.deployments.get_job_id(job_details)

print("[OK] 最適化ジョブが投入されました")
print(f"  Job ID: {job_id}")
print("\n[実行中] ジョブの実行を開始しました...")

[OK] 最適化ジョブが投入されました
  Job ID: 2fd8f277-1907-43e8-9567-5c02bc41456e

[実行中] ジョブの実行を開始しました...


---

## [7] ジョブの監視

### ジョブの状態遷移

```
queued → running → completed
                 ↘ failed
                 ↘ canceled
```

### 各状態の説明

| 状態 | 説明 | 次のアクション |
|------|------|----------------|
| **queued** | キューに入っている（実行待ち） | 待機 |
| **running** | 実行中 | 待機 |
| **completed** | 正常に完了 | 結果を取得 |
| **failed** | エラーで失敗 | エラーログを確認 |
| **canceled** | キャンセルされた | - |


### 監視の仕組み

5秒ごとにジョブの状態をチェックし、完了するまで待機します。



In [12]:
# ジョブの状態を監視（完了するまでループ）
print("=" * 60)
print("ジョブの監視を開始")
print("状態の更新は5秒ごとに行われます")
print("=" * 60)
print()

while True:
    # ジョブの詳細情報を取得
    job_status = client.deployments.get_job_details(job_id)
    
    # 現在の状態を取得
    state = job_status['entity']['decision_optimization']['status']['state']
    
    # 状態を表示
    print(f"[状態] ジョブ状態: {state}")
    
    # 完了状態をチェック
    if state in ["completed", "failed", "canceled"]:
        print()
        print("=" * 60)
        if state == "completed":
            print("[完了] ジョブが正常に完了しました")
        elif state == "failed":
            print("[失敗] ジョブが失敗しました")
        elif state == "canceled":
            print("[中止] ジョブがキャンセルされました")
        print("=" * 60)
        print()
        break
    
    # CPLEXの解法状態が利用可能な場合は表示
    if 'solve_state' in job_status['entity']['decision_optimization']:
        solve_state = job_status['entity']['decision_optimization']['solve_state']
        print(f"  [詳細] 解法状態: {solve_state}")
    
    # 5秒待機
    time.sleep(5)


ジョブの監視を開始
状態の更新は5秒ごとに行われます

[状態] ジョブ状態: queued
[状態] ジョブ状態: queued
[状態] ジョブ状態: queued
[状態] ジョブ状態: queued
[状態] ジョブ状態: queued
[状態] ジョブ状態: queued
[状態] ジョブ状態: queued
[状態] ジョブ状態: running
  [詳細] 解法状態: {'details': {}}
[状態] ジョブ状態: running
  [詳細] 解法状態: {'details': {}, 'latest_engine_activity': []}
[状態] ジョブ状態: running
  [詳細] 解法状態: {'details': {}, 'latest_engine_activity': []}
[状態] ジョブ状態: running
  [詳細] 解法状態: {'details': {}, 'latest_engine_activity': []}
[状態] ジョブ状態: running
  [詳細] 解法状態: {'details': {}, 'latest_engine_activity': []}
[状態] ジョブ状態: running
  [詳細] 解法状態: {'details': {}, 'latest_engine_activity': []}
[状態] ジョブ状態: running
  [詳細] 解法状態: {'details': {}, 'latest_engine_activity': []}
[状態] ジョブ状態: running
  [詳細] 解法状態: {'details': {}, 'latest_engine_activity': []}
[状態] ジョブ状態: running
  [詳細] 解法状態: {'details': {}, 'latest_engine_activity': []}
[状態] ジョブ状態: running
  [詳細] 解法状態: {'details': {}, 'latest_engine_activity': []}
[状態] ジョブ状態: running
  [詳細] 解法状態: {'details': {}, 'latest_engine_activity': ['[2

In [13]:
# 結果表示
import pprint
pprint.pprint(client.deployments.get_job_details(job_id))

{'entity': {'decision_optimization': {'input_data_references': [{'connection': {},
                                                                 'id': 'model.lp.gz',
                                                                 'location': {'href': '/v2/assets/38eb657e-a734-4446-9d1d-911189e2d21f?space_id=53e7afa7-f480-4932-b8e1-62fa18a4a002'},
                                                                 'type': 'data_asset'}],
                                      'output_data': [],
                                      'output_data_references': [{'connection': {},
                                                                  'id': 'solution.xml',
                                                                  'location': {'href': 'https://api.jp-tok.dataplatform.cloud.ibm.com/v2/assets/4dd0bed8-fc03-4664-b349-bc1a557a5f0b?space_id=53e7afa7-f480-4932-b8e1-62fa18a4a002',
                                                                               'id': '4dd0bed8-fc03-

---

## [8] 結果の取得

### 結果ファイルについて

最適化ジョブの実行結果は、自動的にデータアセットとして保存されています。

### solution.xmlの内容

CPLEXの最適化結果がXML形式で保存されています。このファイルには以下の情報が含まれます：

#### 最適化結果
- **目的関数の値**: 最適化された目的関数の値
- **最適性ステータス**: 最適解、実行可能解、非有界など
- **解の品質**: MIPギャップ、相対ギャップなど

#### 変数の値
- **決定変数**: 各変数の最適値
- **スラック変数**: 制約条件のスラック値
- **双対変数**: 線形計画問題の双対解

#### 制約条件の状態
- **制約の充足状況**: 各制約が満たされているか
- **スラック値**: 制約の余裕度
- **双対価格**: 制約の影響度

#### 解法の統計情報
- **実行時間**: 最適化にかかった時間
- **反復回数**: シンプレックス法の反復回数
- **ノード数**: 分枝限定法のノード数（MIPの場合）

### XMLファイルの構造例

```xml
<?xml version="1.0" encoding="UTF-8"?>
<CPLEXSolution version="1.2">
  <header
    problemName="model.lp"
    objectiveValue="12345.67"
    solutionStatusValue="1"
    solutionStatusString="optimal"/>
  <variables>
    <variable name="x1" value="10.5"/>
    <variable name="x2" value="20.3"/>
  </variables>
</CPLEXSolution>
```

In [14]:
# ジョブの最終詳細情報を取得
job_details = client.deployments.get_job_details(job_id)

# 出力データアセットのIDを取得
output_asset_sol_id = job_details['entity']['decision_optimization']["output_data_references"][0]["location"]["id"]
output_asset_log_id = job_details['entity']['decision_optimization']["output_data_references"][1]["location"]["id"]

print("[OK] 出力データアセットIDを取得しました")
print(f"  Output Asset solution ID: {output_asset_sol_id}")
print(f"  Output Asset log ID: {output_asset_log_id}")

[OK] 出力データアセットIDを取得しました
  Output Asset solution ID: 4dd0bed8-fc03-4664-b349-bc1a557a5f0b
  Output Asset log ID: 2df9a18a-3330-4b58-973d-e616b6b047ea


In [15]:
# solution.xmlをダウンロード
client.data_assets.download(
    asset_id=output_asset_sol_id,  # 出力データアセットのID
    filename="solution.xml"     # 保存するファイル名
)

print("[OK] solution.xmlをダウンロードしました")

# log.txtをダウンロード
client.data_assets.download(
    asset_id=output_asset_log_id,  # 出力データアセットのID
    filename="log.txt"     # 保存するファイル名
)

print("[OK] log.txtをダウンロードしました")


Successfully saved data asset content to file: 'solution.xml'
[OK] solution.xmlをダウンロードしました
Successfully saved data asset content to file: 'log.txt'
[OK] log.txtをダウンロードしました


### 結果の解析（オプション）

XMLファイルをPythonで解析する例：

In [16]:
# XMLファイルの解析例（オプション）
import xml.etree.ElementTree as ET

try:
    # XMLファイルを読み込み
    tree = ET.parse('solution.xml')
    root = tree.getroot()
    
    # ヘッダー情報を取得
    header = root.find('header')
    if header is not None:
        print("\n" + "=" * 60)
        print("最適化結果のサマリー")
        print("=" * 60)
        print(f"問題名: {header.get('problemName', 'N/A')}")
        print(f"目的関数値: {header.get('objectiveValue', 'N/A')}")
        print(f"解のステータス: {header.get('solutionStatusString', 'N/A')}")
    
    # 変数の値を表示（最初の10個のみ）
    variables = root.find('variables')
    if variables is not None:
        print("\n" + "=" * 60)
        print("変数の値（最初の10個）")
        print("=" * 60)
        for i, var in enumerate(variables.findall('variable')[:10]):
            name = var.get('name', 'N/A')
            value = var.get('value', 'N/A')
            print(f"{name}: {value}")
        
        total_vars = len(variables.findall('variable'))
        if total_vars > 10:
            print(f"\n... 他 {total_vars - 10} 個の変数")
    
    print("\n[OK] XMLファイルの解析が完了しました")
    
except FileNotFoundError:
    print("[警告] solution.xmlファイルが見つかりません")
except ET.ParseError:
    print("[警告] XMLファイルの解析に失敗しました")
except Exception as e:
    print(f"[警告] エラーが発生しました: {e}")


最適化結果のサマリー
問題名: model.lp.gz
目的関数値: 7
解のステータス: integer optimal solution

変数の値（最初の10個）
x_0: 1
x_1: 1
x_2: 0
x_3: -0

[OK] XMLファイルの解析が完了しました


---

## [9] リソースのクリーンアップ

### クリーンアップの重要性

使用したリソースを削除することで：
1. **コスト削減**: 不要なリソースの課金を防ぐ
2. **スペース管理**: ストレージ容量を節約
3. **整理整頓**: スペースを整理された状態に保つ

### 削除の順序

依存関係を考慮して、以下の順序で削除します：

```
1. 出力データアセット (solution.xmlとlog.txt)
   ↓
2. ジョブ
   ↓
3. 入力データアセット (input_model.lp)
   ↓
4. デプロイメント
   ↓
5. モデル
```



In [17]:
print("=" * 60)
print("リソースのクリーンアップを開始")
print("=" * 60)
print("\n[警告] 削除したリソースは復元できません\n")

リソースのクリーンアップを開始

[警告] 削除したリソースは復元できません



In [18]:
# 1. 出力データアセットの削除
try:
    client.data_assets.delete(asset_id=output_asset_sol_id)
    client.data_assets.delete(asset_id=output_asset_log_id)
    print(f"[OK] [1/5] 出力データアセットを削除しました")
    print(f"        Asset ID: {output_asset_sol_id}")
    print(f"        Asset ID: {output_asset_log_id}")
except Exception as e:
    print(f"[警告] [1/5] 出力データアセットの削除に失敗: {e}")

[OK] [1/5] 出力データアセットを削除しました
        Asset ID: 4dd0bed8-fc03-4664-b349-bc1a557a5f0b
        Asset ID: 2df9a18a-3330-4b58-973d-e616b6b047ea


In [19]:
# 2. ジョブの削除 実際には削除できない。
try:
    client.deployments.delete_job(job_id)
    print(f"[OK] [2/5] ジョブを削除しました")
    print(f"        Job ID: {job_id}")
except Exception as e:
    print(f"[警告] [2/5] ジョブの削除に失敗: {e}")

[OK] [2/5] ジョブを削除しました
        Job ID: 2fd8f277-1907-43e8-9567-5c02bc41456e


In [20]:
# 3. 入力データアセットmodel.lp.gz（LPファイル）の削除
try:
    client.data_assets.delete(model_da_id)
    print(f"[OK] [3/5] 入力データアセットを削除しました")
    print(f"        Asset ID: {model_da_id}")
except Exception as e:
    print(f"[警告] [3/5] 入力データアセットの削除に失敗: {e}")

[OK] [3/5] 入力データアセットを削除しました
        Asset ID: 38eb657e-a734-4446-9d1d-911189e2d21f


In [21]:
# 4. デプロイメントの削除
try:
    client.deployments.delete(deployment_id)
    print(f"[OK] [4/5] デプロイメントを削除しました")
    print(f"        Deployment ID: {deployment_id}")
except Exception as e:
    print(f"[警告] [4/5] デプロイメントの削除に失敗: {e}")

[OK] [4/5] デプロイメントを削除しました
        Deployment ID: 0711fe77-8b26-4530-a507-7829f5be6bce


In [22]:
# 5. モデルの削除
try:
    client.repository.delete(model_id)
    print(f"[OK] [5/5] モデルを削除しました")
    print(f"        Model ID: {model_id}")
except Exception as e:
    print(f"[警告] [5/5] モデルの削除に失敗: {e}")

[OK] [5/5] モデルを削除しました
        Model ID: 6eb66869-f165-4cb7-b40e-794472cf3b22


In [23]:
print("\n" + "=" * 60)
print("[完了] リソースのクリーンアップが完了しました")
print("=" * 60)


[完了] リソースのクリーンアップが完了しました


---

## まとめ

### 完了した処理

このNotebookでは、以下の処理を実行しました：

1. [OK] watsonx.ai Runtime APIクライアントの初期化
2. [OK] Decision Optimizationモデルの登録
3. [OK] デプロイメントの作成（Sサイズ、バッチ処理）
4. [OK] LPファイル（gz圧縮形式）のアップロード
5. [OK] 最適化ジョブの実行（CPLEXソルバー）
6. [OK] ジョブの監視と完了待機
7. [OK] 最適化結果（solution.xml）のダウンロード
8. [OK] 使用したリソースのクリーンアップ

### 学んだこと

- watsonx.ai Runtime Decision Optimization APIの基本的な使い方
- 圧縮ファイル（.gz）の扱い方
- ジョブペイロードの構造と設定方法
- solve_parametersによるCPLEXの制御
- リソース管理とクリーンアップの重要性


### トラブルシューティング

#### ジョブが失敗する場合
```python
# ジョブの詳細情報を確認
job_details = client.deployments.get_job_details(job_id)
print(job_details['entity']['decision_optimization']['status'])
```

#### タイムアウトする場合
```python
# 時間制限を設定
"solve_parameters": {
    "oaas.timeLimit": 3600  # 1時間
}
```


In [24]:
# バージョン情報を表示
import sys
import ibm_watsonx_ai
print(f"Python version: {sys.version}")
print(f"ibm-watsonx-ai version: {ibm_watsonx_ai.__version__}")

Python version: 3.12.10 (tags/v3.12.10:0cc8128, Apr  8 2025, 12:21:36) [MSC v.1943 64 bit (AMD64)]
ibm-watsonx-ai version: 1.4.11
